# Pipe-cutting example

A plumber stocks standard lengths of pipe, all of length 19 m. An order arrives for:

 - 12 lengths of 4m
 - 15 lengths of 5m
 - 22 lengths of 6m

How should these lengths be cut from standard stock pipes so as to minimize
the number of standard pipes used?

Ignore cutting loss; excess pieces may be produced.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are included; no external data files are needed.

### First model: individual stock pipes

Let $x_{ij}$ be the nonnegative integer number of pieces of type $i$ cut
from stock pipe $j$, and let $z_j$ indicate whether that pipe is used.
Each stock pipe has 19 m of capacity. Linking constraints force its piece
counts to zero when $z_j=0$. Minimize $\sum_j z_j$ while meeting demand.

Cutting each requested length separately gives the feasible upper bound
`N` on stock pipes. Change `scale` in both models to compare larger orders.
The commented symmetry-breaking constraints order identical stock pipes;
they can reduce the search without excluding any cutting plan.

In [ ]:
using JuMP, HiGHS

scale = 1

# Cut each requested length separately to obtain a feasible upper bound.
N = cld(12*scale, 4) + cld(15*scale, 3) + cld(22*scale, 3)

m = Model(HiGHS.Optimizer)
set_silent(m)

@variable(m, x[1:3,1:N] >= 0, Int)
@variable(m, z[1:N], Bin)
for j = 1:N
    @constraint(m, 4x[1,j] + 5x[2,j] + 6x[3,j] <= 19)
end
@constraint(m, sum(x[1,j] for j=1:N) >= 12*scale)
@constraint(m, sum(x[2,j] for j=1:N) >= 15*scale)
@constraint(m, sum(x[3,j] for j=1:N) >= 22*scale)
for j = 1:N
    @constraint(m, x[1,j] <= 4z[j])
    @constraint(m, x[2,j] <= 3z[j])
    @constraint(m, x[3,j] <= 3z[j])
end

# # symmetry-breaking
# for j = 1:N-1
#     @constraint(m, z[j] >= z[j+1])
# end

@objective(m, Min, sum(z[j] for j=1:N))

@time(optimize!(m))
if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: " *
        "$(termination_status(m)), $(primal_status(m))")
end

println("number of pipes needed: ",objective_value(m))

### Second model: enumerate cutting patterns

Each column of $A$ is a feasible cutting pattern for one 19-m pipe; its
rows count pieces of lengths 4, 5, and 6 m. The ten supplied patterns are
maximal: none can accept another piece of any requested length.
Because excess pieces are allowed, every nonmaximal pattern can be
extended to a maximal one without increasing the number of stock pipes.

Let $x_p$ be the nonnegative integer number of pipes cut with pattern $p$.
Minimize $\sum_p x_p$ subject to $Ax$ meeting the piece demands.
This example enumerates all maximal patterns in advance.

In [ ]:
using JuMP, HiGHS

# All ten maximal patterns (rows correspond to lengths 4, 5, 6 m):
A = [ 0 0 1 0 2 1 2 3 4 3
      0 1 0 2 1 3 2 1 0 0
      3 2 2 1 1 0 0 0 0 1 ]
scale = 1

m = Model(HiGHS.Optimizer)
set_silent(m)

@variable(m, x[1:size(A,2)] >= 0, Int)
@constraint(m, A*x .>= [12;15;22]*scale)
@objective(m, Min, sum(x))
@time(optimize!(m))
if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: " *
        "$(termination_status(m)), $(primal_status(m))")
end

println("number of pipes needed: ",objective_value(m))
println("number of each pattern:")
println(value.(x))